<a href="https://colab.research.google.com/github/yenhuynh27082003-collab/BUS5001-Q3-ESG-Message-Triage/blob/main/Q3_ESG_Message_Triage_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BUS5001 Q3: ESG Message Triage with an LLM

This notebook runs three ESG operational messages through an API-based large language model and returns structured JSON.

**Important:** Do not place your API key in the notebook or upload it to GitHub.

In [1]:
!pip install -q openai

In [2]:
from getpass import getpass
from openai import OpenAI

api_key = getpass('Enter your OpenAI API key: ')
client = OpenAI(api_key=api_key)
print('Client created successfully.')

Enter your OpenAI API key: ··········
Client created successfully.


## Revised prompt template

The prompt defines fixed labels, requires JSON-only output and prevents unsupported assumptions.

In [3]:
import json

def build_prompt(message):
    return f'''You are an ESG operational message triage assistant.

Classify the employee message below. Return valid JSON only. Do not include Markdown or extra text.

Use exactly these fields:
- issue_category: one of Water, Waste, Energy, Procurement, Accessibility, Governance, Other
- urgency: one of LOW, MEDIUM, HIGH, CRITICAL
- sentiment: one of POSITIVE, NEUTRAL, NEGATIVE
- requires_follow_up: true or false
- recommended_team: one of Facilities, Sustainability, Procurement, Accessibility, Governance, Other
- escalation_required: true or false
- reason: a brief explanation based only on the message

Rules:
1. Do not invent locations, dates, people or impacts.
2. Use HIGH or CRITICAL only when the message provides evidence of serious or immediate risk.
3. Set escalation_required to true for CRITICAL issues or urgent safety/accessibility risks.
4. If the information is unclear, use Other and explain the uncertainty.

Employee message:
{message}

Return JSON only.'''


In [5]:
messages = [
    'There is a water leak in Building C that has been running all morning.',
    'The recycling bins are contaminated again and no one seems to be checking them.',
    'The accessible entrance near the main building has been blocked for two days.'
]

results = []

for message in messages:
    response = client.responses.create(
        model='gpt-4.1-mini',
        input=build_prompt(message)
    )

    result = json.loads(response.output_text)
    results.append({'message': message, 'llm_output': result})

    print('MESSAGE:', message)
    print(json.dumps(result, indent=2))
    print('-' * 70)

MESSAGE: There is a water leak in Building C that has been running all morning.
{
  "issue_category": "Water",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "requires_follow_up": true,
  "recommended_team": "Facilities",
  "escalation_required": false,
  "reason": "Water leak in Building C running all morning indicates a significant maintenance issue requiring prompt attention."
}
----------------------------------------------------------------------
MESSAGE: The recycling bins are contaminated again and no one seems to be checking them.
{
  "issue_category": "Waste",
  "urgency": "MEDIUM",
  "sentiment": "NEGATIVE",
  "requires_follow_up": true,
  "recommended_team": "Facilities",
  "escalation_required": false,
  "reason": "Recycling bins are contaminated and not being checked, indicating ongoing waste management issue."
}
----------------------------------------------------------------------
MESSAGE: The accessible entrance near the main building has been blocked for two days.
{

In [ ]:
with open('q3_llm_outputs.json', 'w', encoding='utf-8') as file:
    json.dump(results, file, indent=2, ensure_ascii=False)

print('Saved results to q3_llm_outputs.json')

In [6]:
!pip install -q transformers torch pandas

In [7]:
from transformers import pipeline
import pandas as pd

hf_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

candidate_labels = [
    "Water",
    "Waste",
    "Energy",
    "Procurement",
    "Accessibility",
    "Governance",
    "Other"
]

print("Hugging Face classifier is ready.")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Hugging Face classifier is ready.


In [8]:
hf_results = []

for item in results:
    message = item["message"]

    prediction = hf_classifier(
        message,
        candidate_labels,
        multi_label=False
    )

    hf_category = prediction["labels"][0]
    confidence = prediction["scores"][0]

    hf_results.append({
        "message": message,
        "llm_category": item["llm_output"]["issue_category"],
        "huggingface_category": hf_category,
        "huggingface_confidence": round(confidence, 3),
        "consistent": (
            item["llm_output"]["issue_category"] == hf_category
        )
    })

comparison_df = pd.DataFrame(hf_results)
comparison_df

,message,llm_category,huggingface_category,huggingface_confidence,consistent
0,There is a water leak in Building C that has b...,Water,Water,0.783,True
1,The recycling bins are contaminated again and ...,Waste,Waste,0.392,True
2,The accessible entrance near the main building...,Accessibility,Accessibility,0.900,True
